## EDA marathos - silver 
- cleaning


In [0]:
df = spark.sql("FROM marathos.bronze.raw_marathos")
df.display()

In [0]:
df.columns

In [0]:
import re 

#tested to do both functions but will use snake_case for good readibility 

def to_snake_case(name):
    snakecase = snakecase =re.sub(r"[\s]+", "_", name.strip().lower())
    return snakecase

def to_camelCase(name): 
    snakecase = to_snake_case(name)
    words = snakecase.split("_")
    camelcase = words[0].lower() + "".join(word.capitalize() for word in words[1:])
    return camelcase


In [0]:
to_snake_case("Athlete club")

In [0]:
to_camelCase("Athlete club")

In [0]:
def rename_columns_to_snakecase(df): 
    new_columns = [to_snake_case(column) for column in df.columns]
    return df.toDF(*new_columns) 

In [0]:
df_cleaned_columns = rename_columns_to_snakecase(df)
df_cleaned_columns.display()

In [0]:
for column in df.columns: 
    print(column, df.filter(df[column].isNull()).count())

In [0]:
from pyspark.sql.functions import to_timestamp, col, coalesce, lit, when, round as spark_round, lower, trim

In [0]:
df_cleaned_columns.select("event_distance/length", "athlete_performance").limit(30).display()

In [0]:
df_cleaned_columns.select("athlete_performance","athlete_average_speed","event_number_of_finishers").orderBy("athlete_average_speed").display()


In [0]:
def clean_null_values_string(df):
    for column, col_type in df.dtypes:
        if col_type == "string":
            df = df.withColumn(
                column,
                trim(regexp_replace(col(column), r"^[#+*]+|[#+*]+$", ""))
            )
            
            df = df.withColumn(
                column, 
                when(
                    col(column).isNull()
                    | (trim(col(column)) == "")
                    | (lower(trim(col(column))) == "null"),
                    lit("unknown"),
                ).otherwise(trim(col(column))),
            )
    return df


df_cleaned = clean_null_values_string(df_cleaned_columns)
df_cleaned.display()

In [0]:
from pyspark.sql.functions import (
    try_to_date,
    round as spark_round,
    trim,
    regexp_replace,
)

# Cleaned dataframe
df_cleaned = (
    df_cleaned_columns.withColumn(
        "athlete_year_of_birth",
        coalesce(
            col("athlete_year_of_birth").cast("int").cast("string"), lit("unknown")
        ),
    )
    .withColumn("event_dates", try_to_date(trim(col("event_dates")), "dd.MM.yyyy"))
    .withColumn(
        "event_unit",
        when(col("event_distance/length").contains("km"), "km")
        .when(col("event_distance/length").contains("mi"), "mi")
        .when(col("event_distance/length").contains("h"), "h")
        .otherwise("unknown"),
    )
    .withColumn(
        "performance_unit",
        when(col("athlete_performance").contains("km"), "km")
        .when(col("athlete_performance").contains("h"), "h")
        .otherwise("unknown"),
    )
    .withColumn(
        "is_valid_performance",
        when(col("athlete_performance").contains("d"), False)
        .when(
            col("event_unit").isin("km", "mi") & (col("performance_unit") == "h"), True
        )
        .when(col("event_unit").isin("h") & (col("performance_unit") == "km"), True)
        .otherwise(False),
    )
    .filter(col("is_valid_performance") == True)
    .withColumn(
        "athlete_performance_distance_km",
        when(
            col("athlete_performance").contains("km"),
            col("athlete_performance").cast("double"),
        ).otherwise(None),
    )
   .withColumn(
        "athlete_performance_distance_km",
        when(
            col("performance_unit") == "km", 
            regexp_replace(col("athlete_performance"), "[^0-9.]", "").cast("double")
        ).otherwise(None)
    )
    .withColumn(
        "performance_time",
        when(
            col("performance_unit") == "h", 
            trim(regexp_replace(col("athlete_performance"), "h", ""))
        ).otherwise(None)
    )
)

df_cleaned.select(
    "event_name", 
    "event_dates",
   "event_distance/length",          
    "event_unit",                     
    "athlete_performance",            
    "performance_unit",               
    "athlete_performance_distance_km",   
    "performance_time",         
    "athlete_year_of_birth"
).display()

In [0]:
w_event = Window.orderBy("event_name")

w_race = Window.partitionBy("event_name").orderBy("event_dates")

df_cleaned = (
    df_cleaned
    
    .withColumn("event_id", dense_rank().over(w_event))
    
    .withColumn("race_id", dense_rank().over(w_race))
)

In [0]:
df_cleaned.display()